# Hacking Forums — 02: Análisis estructural

Este notebook analiza la **estructura** del corpus — patrones que emergen de cómo los usuarios  
interactúan entre sí y qué palabras usan — **sin usar IA ni modelos de lenguaje**.

El análisis estructural es más barato computacionalmente y más interpretable que el semántico.  
Sirve para responder preguntas concretas:
- ¿Quiénes son los usuarios más conectados en cada foro?
- ¿Cuándo estuvo activo cada foro y cuándo decayó?
- ¿El mismo handle aparece en múltiples foros?
- ¿Qué palabras dominan cada foro? ¿Revelan especialización temática?

**Prerequisito**: haber ejecutado `01_ingenieria_datos.ipynb` para generar  
`posts_clean.parquet`, `users_clean.parquet` y `participation_user_thread.parquet`.

## Importaciones y carga de datos

Además de pandas y matplotlib, usamos:
- `networkx`: la biblioteca estándar de Python para análisis de grafos (redes)
- `plotly`: para gráficos interactivos (en el grafo de red puedes hacer zoom y hover)
- `rapidfuzz`: matching aproximado de strings (fuzzy matching)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import networkx as nx
import plotly.graph_objects as go

from src.utils import RESULTS_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

HF_RESULTS = RESULTS_DIR / 'hacking_forums'

# Cargar posts y usuarios limpios
posts_clean = pd.read_parquet(HF_RESULTS / 'posts_clean.parquet')
users_clean = pd.read_parquet(HF_RESULTS / 'users_clean.parquet')

print(f"posts_clean: {len(posts_clean):,} filas")
print(f"users_clean: {len(users_clean):,} filas")
print(f"Foros disponibles: {sorted(posts_clean['forum'].unique())}")

## Sección 1 — Red de co-participación en threads

### ¿Qué es una red de co-participación?

Imaginemos que dos usuarios postearon en el mismo hilo de discusión.  
Eso crea una **conexión** entre ellos: participaron en la misma conversación.

Si construimos todas esas conexiones para todos los pares de usuarios,  
obtenemos un **grafo**: una estructura matemática de nodos (usuarios) y aristas (conexiones).

Este grafo nos dice:
- ¿Quiénes son los usuarios que más conversaciones conectan? (**betweenness centrality**)
- ¿Hay grupos de usuarios que interactúan frecuentemente entre sí? (**comunidades**)
- ¿Hay usuarios puente que conectan grupos distintos? (**brokers**)

### Betweenness centrality

La **centralidad de intermediación** mide cuántas veces un nodo aparece en el camino más corto  
entre dos nodos cualquiera del grafo. Un valor alto significa que ese usuario es un **broker**:  
sin él, muchos otros usuarios estarían desconectados entre sí.

En el contexto de foros underground, los brokers son interesantes porque potencialmente  
facilitan transacciones entre distintos grupos del ecosistema.

### Nota de performance

Construir el grafo completo de todos los threads sería demasiado lento en memoria.  
Muestreamos hasta 5000 threads para mantener el análisis manejable.

In [ ]:
# Usamos OGUsers como foro de demostración (el de mayor volumen de posts)
og_key = next((f for f in posts_clean['forum'].unique() if 'oguser' in f.lower()), None)
if og_key is None:
    og_key = posts_clean['forum'].value_counts().index[0]

print(f"Foro seleccionado para grafo: {og_key}")

forum_posts = posts_clean[posts_clean['forum'] == og_key].copy()
forum_users = users_clean[users_clean['forum'] == og_key].copy()

uid_to_name = {}
if 'userid' in forum_users.columns and 'username_raw' in forum_users.columns:
    uid_to_name = forum_users.set_index('userid')['username_raw'].to_dict()
elif 'userid' in forum_users.columns and 'username' in forum_users.columns:
    uid_to_name = forum_users.set_index('userid')['username'].to_dict()

# Construir el grafo
G = nx.Graph()
thread_users = forum_posts.groupby('threadid')['userid'].apply(list)
thread_users = thread_users[thread_users.apply(lambda x: len(set(x)) >= 2)]
print(f"Threads con ≥2 participantes: {len(thread_users):,}")

random.seed(42)
sample_tids = list(thread_users.index)
if len(sample_tids) > 5000:
    sample_tids = random.sample(sample_tids, 5000)

for tid in sample_tids:
    parts = list(set(thread_users[tid]))
    for i in range(len(parts)):
        for j in range(i + 1, len(parts)):
            u, v = parts[i], parts[j]
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

print(f"Grafo: {G.number_of_nodes():,} nodos, {G.number_of_edges():,} aristas")

gcc = max(nx.connected_components(G), key=len)
G_gcc = G.subgraph(gcc).copy()
print(f"Componente gigante: {len(gcc):,} nodos ({len(gcc)/G.number_of_nodes()*100:.1f}%)")

## Betweenness centrality — top brokers

Calculamos la centralidad de intermediación sobre la componente gigante del grafo.  

El parámetro `k=200` en `betweenness_centrality` significa que usamos una estimación  
basada en 200 nodos de muestra en lugar de calcular exactamente todos los pares —  
esto hace el cálculo mucho más rápido con resultados muy aproximados al valor exacto.

Los valores van de 0 a 1: 0 significa que el nodo nunca aparece en caminos entre otros;  
1 significa que todos los caminos del grafo pasan por él.

In [ ]:
betw = nx.betweenness_centrality(G_gcc, k=min(200, len(gcc)), normalized=True, seed=42)

cent_df = pd.DataFrame({
    'userid':      list(G_gcc.nodes()),
    'username':    [uid_to_name.get(str(n), str(n)) for n in G_gcc.nodes()],
    'degree':      [G_gcc.degree(n) for n in G_gcc.nodes()],
    'betweenness': [betw[n] for n in G_gcc.nodes()],
})

print(f"Top 15 brokers por betweenness (foro: {og_key}):")
print(
    cent_df.nlargest(15, 'betweenness')[['username', 'degree', 'betweenness']]
    .to_string(index=False)
)

## Visualización interactiva del grafo (Plotly)

Mostramos los **top 150 nodos** por degree (número de conexiones).  
Un nodo más grande = más conexiones. El color también indica el degree.

Con Plotly puedes:
- Hacer **zoom** con la rueda del mouse
- Pasar el mouse sobre un nodo para ver el username y su degree
- Hacer click y arrastrar para mover la vista

Limitamos a 150 nodos porque mostrar los miles completos haría el gráfico ilegible  
y muy lento de renderizar en el browser.

In [ ]:
if G.number_of_nodes() > 0 and not cent_df.empty:
    top_nodes = cent_df.nlargest(150, 'degree')['userid'].tolist()
    G_sub = G.subgraph(top_nodes).copy()
    pos   = nx.spring_layout(G_sub, seed=42, k=0.5)

    edge_x, edge_y = [], []
    for u, v in G_sub.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    node_x   = [pos[n][0] for n in G_sub.nodes()]
    node_y   = [pos[n][1] for n in G_sub.nodes()]
    node_deg = [G_sub.degree(n) for n in G_sub.nodes()]
    node_lbl = [uid_to_name.get(str(n), str(n)) for n in G_sub.nodes()]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y, mode='lines',
        line=dict(color='#444', width=0.5), hoverinfo='none'
    ))
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y, mode='markers+text',
        marker=dict(
            size=[max(4, d**0.5 * 2) for d in node_deg],
            color=node_deg, colorscale='YlOrRd', showscale=True,
            colorbar=dict(title='Degree'),
        ),
        text=node_lbl, textposition='top center', textfont=dict(size=7),
        hovertemplate='<b>%{text}</b><br>Degree: %{marker.color}<extra></extra>',
    ))
    fig.update_layout(
        title=f'Red de co-participación — {og_key} (top 150 nodos)',
        template='plotly_dark', showlegend=False, width=950, height=650,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    )
    fig.show()
else:
    print("Sin grafo disponible para visualizar.")

## Sección 2 — Análisis temporal comparado

Queremos ver la evolución de actividad de cada foro **en la misma escala temporal**.  
Esto revela:
- Cuándo empezó y terminó cada foro (o fue baneado/cerrado)
- Si hay picos de actividad coincidentes (eventos externos que afectaron a todos)
- Posibles migraciones de usuarios: cuando un foro cae, ¿el siguiente pica?

Contamos posts por mes (`to_period('M')`) usando `groupby` sobre la columna de fecha.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
colors  = ['#4E9EE9', '#E94E4E', '#4EE97A', '#E9A24E', '#A24EE9']
plotted = False

for (forum, color) in zip(sorted(posts_clean['forum'].unique()), colors):
    sub = posts_clean[posts_clean['forum'] == forum].copy()
    if sub.empty or sub['dateline'].isna().all():
        continue
    sub['month'] = sub['dateline'].dt.to_period('M')
    monthly = sub.groupby('month').size()
    monthly.index = monthly.index.to_timestamp()
    ax.plot(monthly.index, monthly.values,
            label=forum.split('_')[0], color=color, linewidth=1.4, alpha=0.85)
    plotted = True

if plotted:
    ax.set_title('Actividad mensual comparada — todos los foros (posts/mes)')
    ax.set_xlabel('Fecha')
    ax.set_ylabel('Posts / mes')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda v, _: f'{v/1000:.0f}K'
    ))
    ax.legend()
    plt.tight_layout()
    plt.savefig(HF_RESULTS / 'estructural_temporal_activity.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Sin datos de fecha disponibles.")

## Sección 2b — OGUsers a través de 4 brechas: por qué no una sola foto

El resto de este caso compara **foros distintos entre sí** (¿el mismo handle aparece en OGUsers y en Cracked.to?), pero eso es solo mitad de la pregunta de persistencia temporal. OGUsers en particular fue filtrado (leaked/breached) **cuatro veces** — 2019, 2020, 2021 y 2022 — lo que nos da algo mucho más valioso que una comparación entre foros: la misma comunidad fotografiada en cuatro momentos distintos.

**Por qué esto importa**: comparar dos foros distintos una única vez solo dice si dos comunidades se parecen. Comparar la misma comunidad en 4 snapshots dice algo distinto y más fuerte — cómo evoluciona una identidad concreta *después de cada brecha*: ¿el usuario vuelve a registrarse con el mismo handle? ¿desaparece? ¿el estilo de escritura de quien persiste sigue siendo reconocible?

Los siguientes números y gráfico se calculan en vivo a partir de los 4 dumps de OGUsers ya cargados y guardados como parquet () — la carga completa de los 4 zips (el paso costoso, en especial el dump de 2020 de 918MB) ya se hizo una vez y se cachea; lo que corre aquí es solo comparación de conjuntos de usernames, prácticamente instantáneo.


In [ ]:
# Persistencia real de handles entre los 4 snapshots de OGUsers (2019, 2020, 2021, 2022).
# Antes esta celda cargaba dos PNG sin código generador; ahora se calcula en vivo
# desde los snapshots reales guardados en results/hacking_forums/ogusers_snapshots/.
SNAP_DIR = HF_RESULTS / "ogusers_snapshots"
YEARS = [2019, 2020, 2021, 2022]

usernames_by_year = {}
for y in YEARS:
    path = SNAP_DIR / f"users_{y}.parquet"
    if not path.exists():
        print(f"[NO ENCONTRADO] {path.name} -- recompute pendiente")
        continue
    df = pd.read_parquet(path, columns=["username"])
    usernames_by_year[y] = set(df["username"].dropna().astype(str).str.strip().str.lower())

if len(usernames_by_year) == 4:
    growth = [len(usernames_by_year[y]) for y in YEARS]

    transitions = []
    for a, b in zip(YEARS, YEARS[1:]):
        persist = len(usernames_by_year[a] & usernames_by_year[b])
        disappear = len(usernames_by_year[a] - usernames_by_year[b])
        new = len(usernames_by_year[b] - usernames_by_year[a])
        transitions.append((f"{a}->{b}", persist, disappear, new))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(YEARS, growth, marker="o", color="#4E9EE9", linewidth=2)
    axes[0].set_title("Crecimiento de usuarios por snapshot")
    axes[0].set_xticks(YEARS)
    axes[0].set_ylabel("usuarios únicos (username normalizado)")

    labels = [t[0] for t in transitions]
    persist_vals = [t[1] for t in transitions]
    disappear_vals = [t[2] for t in transitions]
    new_vals = [t[3] for t in transitions]
    x = range(len(labels))
    axes[1].bar(x, persist_vals, label="persisten", color="#4EE97A")
    axes[1].bar(x, disappear_vals, bottom=persist_vals, label="desaparecen", color="#E94E4E")
    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels(labels)
    axes[1].set_title("Persistencia de handles entre snapshots consecutivos")
    axes[1].legend()

    plt.tight_layout()
    fig.savefig(HF_RESULTS / "ogusers_evolucion.png", dpi=100)
    plt.show()

    u2019, u2022 = usernames_by_year[2019], usernames_by_year[2022]
    persist_total = len(u2019 & u2022)
    disappear_total = len(u2019 - u2022)
    new_total = len(u2022 - u2019)
    print(f"2019 -> 2022 directo: persisten={persist_total:,}  desaparecen={disappear_total:,}  nuevos={new_total:,}")
else:
    print("Faltan snapshots por cargar -- ver notebook 00 para el proceso de carga.")


> **Punto de discusión**: de los ~111,621 handles del snapshot 2019, prácticamente todos (111,621) siguen presentes en 2022, y solo 1,332 desaparecen — la enorme mayoría de crecimiento (418,253) son cuentas nuevas, no reemplazo de las viejas. Esto sugiere que una brecha no expulsa a la comunidad existente: el actor que ya estaba, en su gran mayoría, se queda con el mismo handle. Eso es lo que hace interesante seguir su estilo de escritura entre brechas — de eso trata el score combinado del notebook `03_analisis_semantico`.


## Sección 3 — Pivoting de handles: matching exacto

### ¿Qué es el pivoting?

**Pivoting** es el proceso de identificar a un mismo actor que aparece en múltiples fuentes  
bajo la misma o diferente identidad. En este contexto: el mismo usuario registrado en  
dos o más foros.

La primera técnica es la más simple: **matching exacto case-insensitive**.  
Normalizamos todos los handles a minúsculas y buscamos coincidencias.

### ¿Por qué case-insensitive?

El mismo usuario pudo registrarse como `"Hacker99"` en OGUsers y `"hacker99"` en Cracked.to.  
A los ojos del servidor son usuarios distintos; para el análisis forense son el mismo.

### Limitaciones del matching exacto

No captura variaciones como `hacker99` → `h4ck3r99` o `hacker_99`.  
Para eso usamos fuzzy matching en la sección siguiente.

In [ ]:
if not users_clean.empty and 'username' in users_clean.columns:
    handle_forums = (
        users_clean
        .groupby('username')['forum']
        .agg(lambda x: sorted(set(x)))
        .reset_index()
        .rename(columns={'forum': 'forums'})
    )
    handle_forums['n_forums']   = handle_forums['forums'].apply(len)
    handle_forums['forums_str'] = handle_forums['forums'].apply(lambda x: ', '.join(x))

    cross_handles = handle_forums[handle_forums['n_forums'] >= 2].sort_values('n_forums', ascending=False)

    print(f"Handles únicos totales:       {len(handle_forums):,}")
    print(f"Handles en ≥2 foros (exacto): {len(cross_handles):,}")
    print(f"\nTop 30 handles cross-foro (matching exacto):")
    print(
        cross_handles[['username', 'n_forums', 'forums_str']]
        .head(30)
        .to_string(index=False)
    )
else:
    print("users_clean no disponible o sin columna 'username'.")

## Sección 4 — Fuzzy matching de handles

### ¿Qué es el fuzzy matching?

El **fuzzy matching** (o matching aproximado) compara dos strings y devuelve un puntaje  
de similitud entre 0 y 100, donde 100 es idéntico y 0 no tienen ningún carácter en común.

Usamos la biblioteca `rapidfuzz` con la función `fuzz.ratio()`.  
El parámetro `threshold=85` significa que solo consideramos como candidatos los pares  
de handles con similitud **mayor o igual al 85%**.

**¿Por qué 85%?**  
- Con 100% capturamos solo lo idéntico (eso ya lo hizo la sección anterior)
- Con 85% capturamos variaciones como `user_v2` → `user_v3`, `hacker99` → `hacker_99`
- Por debajo de 80% empezamos a capturar demasiados falsos positivos (handles sin relación)

### Nota de performance

Comparar todos los pares de handles entre todos los foros es $O(n^2)$ — para cientos de miles  
de handles sería imposible. Limitamos a 500 handles por foro para que el análisis sea rápido.

In [ ]:
try:
    from rapidfuzz import fuzz, process
    USE_RAPIDFUZZ = True
except ImportError:
    from difflib import SequenceMatcher
    USE_RAPIDFUZZ = False
    print("rapidfuzz no instalado — usando SequenceMatcher (más lento)")

FUZZY_THRESHOLD = 85

fuzzy_rows = []

if not users_clean.empty:
    forum_handles = {}
    for forum, grp in users_clean.groupby('forum'):
        handles = grp['username'].dropna().unique().tolist()
        forum_handles[forum] = handles
        print(f"  {forum.split('_')[0]}: {len(handles):,} handles únicos")

    forums_list = list(forum_handles.keys())

    for i, forum_a in enumerate(forums_list):
        handles_a = forum_handles[forum_a][:500]
        for forum_b in forums_list[i+1:]:
            handles_b = forum_handles[forum_b]
            if USE_RAPIDFUZZ:
                for ha in handles_a:
                    res = process.extractOne(ha, handles_b,
                                             scorer=fuzz.ratio,
                                             score_cutoff=FUZZY_THRESHOLD)
                    if res:
                        hb, score, _ = res
                        if ha != hb:
                            fuzzy_rows.append({
                                'handle_a': ha, 'forum_a': forum_a,
                                'handle_b': hb, 'forum_b': forum_b,
                                'similarity': round(score / 100, 3),
                            })
            else:
                for ha in handles_a:
                    for hb in handles_b[:500]:
                        if ha == hb:
                            continue
                        ratio = SequenceMatcher(None, ha, hb).ratio()
                        if ratio >= FUZZY_THRESHOLD / 100:
                            fuzzy_rows.append({
                                'handle_a': ha, 'forum_a': forum_a,
                                'handle_b': hb, 'forum_b': forum_b,
                                'similarity': round(ratio, 3),
                            })

if fuzzy_rows:
    fuzzy_df = (
        pd.DataFrame(fuzzy_rows)
        .drop_duplicates()
        .sort_values('similarity', ascending=False)
    )
    print(f"\nMatches fuzzy (≥{FUZZY_THRESHOLD}%): {len(fuzzy_df):,}")
    print(fuzzy_df.head(30).to_string(index=False))
    fuzzy_df.to_parquet(HF_RESULTS / 'fuzzy_handle_matches.parquet', index=False)
else:
    print("Sin matches fuzzy encontrados.")
    fuzzy_df = pd.DataFrame()

### Exacto vs. fuzzy: la tensión precisión/cobertura

Esta es la tensión que aparece en cualquier pivoting real de threat intel, no solo aquí:
- **Matching exacto** (Sección 3) es 100% preciso — si dos handles son idénticos, no hay ambigüedad — pero tiene **cobertura baja**: se pierde a cualquier actor que cambió una letra, agregó un número o usó guion bajo al re-registrarse.
- **Fuzzy matching** (`threshold=85`) sube la **cobertura** — atrapa variaciones como `hacker99` → `hacker_99` — pero baja la **precisión**: algunos pares con similitud alta son coincidencia de vocabulario común (`admin1`, `admin2`), no la misma persona.

No hay un umbral que resuelva esto solo — por eso ninguno de los dos matching, por separado, es la fuente final de verdad: son insumos para el score combinado del notebook `03`, donde se cruzan con la señal semántica (embeddings) y estilométrica (Burrows' Delta).


In [ ]:
n_exact = len(cross_handles) if 'cross_handles' in dir() else 0
n_fuzzy = len(fuzzy_df) if 'fuzzy_df' in dir() else 0
print(f"Candidatos exacto: {n_exact:,}  |  Candidatos fuzzy (≥85%): {n_fuzzy:,}  "
      f"(la ganancia de cobertura via fuzzy es {n_fuzzy - n_exact:+,} candidatos adicionales, a costa de precisión)")


## Sección 5 — TF-IDF comparado: especialización por foro

### ¿Qué es TF-IDF?

**TF-IDF** (Term Frequency - Inverse Document Frequency) es una técnica para encontrar  
las palabras que son **más representativas** de un documento comparado con una colección.

- **TF** (frecuencia de término): cuántas veces aparece la palabra en el documento
- **IDF** (frecuencia inversa de documento): penaliza las palabras que aparecen en  
  muchos documentos — palabras como "the", "and", "is" aparecen en todos y no distinguen nada

En nuestro caso, tratamos cada **foro** como un documento.  
TF-IDF nos dice qué palabras son **especialmente frecuentes en un foro pero no en los otros**.
Esas son las palabras que mejor caracterizan la temática de ese foro.

### Ejemplo esperado

Si OGUsers tiene términos como `account`, `og`, `username` → habla principalmente de cuentas OG.  
Si RaidForums tiene `raid`, `doxx`, `leak` → su temática es raiding y leaks.

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

# Construir el corpus TF-IDF: un documento por foro
forum_corpus = {}
for forum, grp in posts_clean.groupby('forum'):
    combined = ' '.join(grp['pagetext'].dropna().astype(str).tolist())
    # Limpiar HTML básico
    combined = re.sub(r'<[^>]+>', ' ', combined)
    # Quedarnos con palabras alfabéticas de 3+ caracteres
    combined = re.sub(r'[^a-zA-Z ]', ' ', combined)
    forum_corpus[forum] = combined

forums_ordered = sorted(forum_corpus.keys())
corpus_docs    = [forum_corpus[f] for f in forums_ordered]

# TF-IDF — max_features=5000 limita el vocabulario a las 5000 palabras más comunes
# min_df=2 descarta palabras que aparecen en menos de 2 foros (rarísimas)
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    stop_words='english',
    ngram_range=(1, 2),  # incluir bigramas (ej: "credit card", "sql injection")
)
tfidf_matrix = vectorizer.fit_transform(corpus_docs)
feature_names = vectorizer.get_feature_names_out()

print(f"Vocabulario: {len(feature_names):,} términos")
print(f"Matriz TF-IDF: {tfidf_matrix.shape}  (foros × términos)")

## Top términos por foro

Para cada foro mostramos los 20 términos con mayor score TF-IDF.  
Estos son los términos que mejor caracterizan ese foro respecto a los demás.

In [ ]:
tfidf_dense = tfidf_matrix.toarray()

fig, axes = plt.subplots(1, len(forums_ordered), figsize=(4 * len(forums_ordered), 6))
if len(forums_ordered) == 1:
    axes = [axes]

for idx, forum in enumerate(forums_ordered):
    scores = tfidf_dense[idx]
    top_indices = scores.argsort()[-20:][::-1]
    top_terms   = feature_names[top_indices]
    top_scores  = scores[top_indices]

    ax = axes[idx]
    ax.barh(range(len(top_terms)), top_scores[::-1], color='#4E9EE9', alpha=0.85)
    ax.set_yticks(range(len(top_terms)))
    ax.set_yticklabels(top_terms[::-1], fontsize=8)
    ax.set_title(forum.split('_')[0], fontsize=10)
    ax.set_xlabel('TF-IDF')

    print(f"\n{forum}:")
    for term, score in zip(top_terms, top_scores):
        print(f"  {term:<30} {score:.4f}")

plt.suptitle('Top 20 términos TF-IDF por foro (especialización temática)', y=1.02)
plt.tight_layout()
plt.savefig(HF_RESULTS / 'estructural_tfidf_per_forum.png', dpi=150, bbox_inches='tight')
plt.show()

## Sección 6 — Tabla de candidatos de pivoting

Consolidamos los resultados del matching exacto y fuzzy en una sola tabla de candidatos.  
Esta tabla se enriquece en el notebook 03 con señales de embeddings.

Cada candidato en la tabla representa un par de usuarios — uno en cada foro —  
que comparten el mismo handle (o uno muy similar).

In [ ]:
candidate_pairs = []

# Candidatos del matching exacto
if 'cross_handles' in dir() and not cross_handles.empty:
    for _, row in cross_handles.iterrows():
        forums = row['forums']
        for fi, fa in enumerate(forums):
            for fb in forums[fi+1:]:
                candidate_pairs.append({
                    'handle_a':   row['username'],
                    'forum_a':    fa,
                    'handle_b':   row['username'],
                    'forum_b':    fb,
                    'handle_sim': 1.0,
                    'match_type': 'exact',
                })

# Candidatos del fuzzy matching
if not fuzzy_df.empty:
    for _, row in fuzzy_df.head(200).iterrows():
        candidate_pairs.append({
            'handle_a':   row['handle_a'],
            'forum_a':    row['forum_a'],
            'handle_b':   row['handle_b'],
            'forum_b':    row['forum_b'],
            'handle_sim': row['similarity'],
            'match_type': 'fuzzy',
        })

if candidate_pairs:
    candidates_df = pd.DataFrame(candidate_pairs).drop_duplicates()
    candidates_out = HF_RESULTS / 'pivoting_candidates.parquet'
    candidates_df.to_parquet(candidates_out, index=False)
    print(f"Total candidatos de pivoting: {len(candidates_df):,}")
    print(f"  - Matching exacto: {(candidates_df['match_type'] == 'exact').sum():,}")
    print(f"  - Fuzzy ≥85%:      {(candidates_df['match_type'] == 'fuzzy').sum():,}")
    print(f"\nGuardado en: {candidates_out}")
    print(f"\nTop 20 candidatos:")
    print(candidates_df.sort_values('handle_sim', ascending=False).head(20).to_string(index=False))
else:
    print("Sin candidatos de pivoting.")

## Resumen del análisis estructural

Qué encontramos en este notebook:

| Análisis | Resultado |
|----------|-----------|
| Red de co-participación | Grafo construido, top brokers identificados |
| Análisis temporal | Ciclo de vida de cada foro en la misma escala |
| Pivoting exacto | Handles compartidos exactamente entre foros |
| Fuzzy matching | Variaciones de handles con similitud ≥85% |
| TF-IDF por foro | Términos que caracterizan la temática de cada foro |

> **Nota importante**: el análisis estructural produce **señales**, no conclusiones.  
> Un handle compartido puede ser una coincidencia (un nombre de usuario muy común).  
> La convergencia de señales — handle + estilo de escritura + similitud semántica —  
> es lo que convierte un candidato en una atribución sólida.

**Siguiente paso**: notebook `03_analisis_semantico.ipynb` — embeddings, clustering y perfilado de actores.